# Lineární klasifikace obrázků CIFAR10

- Úkolem cvičení je naprogramovat lineární klasifikátor, který bude rozpoznávat objekty z datasetu CIFAR-10.
- Vstupem je RGB obrázek o rozměrech 32×32 pixelů a úkolem říci, který z 10 možných objektů (tříd) je na něm zachycen.
- Možnosti jsou tyto: letadlo, automobil, pták, kočka, jelen, pes, žába, kůň, loď, náklaďák.
- Použijeme přitom **lineární model**, **softmax křížovou entropii** a **gradient descent**.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")  # import tests, ans

import matplotlib.pyplot as plt
import numpy as np
import PIL
import torch
import torchvision
import os 

import ans
from tests import test_linear_classification

# Data

## Preprocessing

- Data převedeme z `PIL.Image` do `torch.Tensor`.
- Pro lepší numerické vlastnosti zároveň konvertujeme do rozsahu 0-1 a datového typu `torch.float32`, který je pro PyTorch výchozí. Stačí vydělit 255.
- Zároveň obrázky přetvarujeme do vektoru, takže výstup bude mít rozměr `(batch_size, počet_pixelů)`.
- Targety (tj. anotace) pouze převedeme z `numpy.ndarray` do `torch.Tensor`.
- Celé předzpracování bude implementovat funkce `preprocess`, kterou budeme později volat pro každou dávku po jejím načtení v `BatchLoader`u.

### TODO: implementuje funkci `preprocess`.

In [3]:
def preprocess(img: PIL.Image) -> torch.Tensor:
    """
    Args:
        img: image of type PIL.Image
    Returns:
        x: 1-dimensional tensor; shape (num_pixels * num_channels,), dtype float32
    """

    ########################################
    # TODO: implement

    shape = np.array(img)
    x = torch.tensor(shape, dtype = torch.float32) / 255.0
    x = x.reshape(-1)

    # ENDTODO
    ########################################

    return x

In [4]:
test_linear_classification.TestPreprocess.eval(preprocess_fn=preprocess)

test_preprocess (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_preprocess) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Trénovací a validační datasety

In [5]:
train_dataset = torchvision.datasets.CIFAR10(root="../data", train=True, download=True, transform=preprocess)
val_dataset = torchvision.datasets.CIFAR10(root="../data", train=False, download=True, transform=preprocess)

In [6]:
train_dataset[5]

(tensor([0.6235, 0.4000, 0.3961,  ..., 0.7137, 0.2235, 0.0745]), 1)

## Načítání po dávkách

- Načítání dat po dávkách bude zajišťovat třída `ans.data.BatchLoader`.
- Třída implementuje metodu `__iter__` a je tedy možné ji použít jako zdroj např. pro `for` cyklus následujícím způsobem:
  ``` python
  train_loader = ans.data.BatchLoader(
      dataset,  # napr. torchvision.datasets.CIFAR10
      batch_size = 100,
      shuffle = True,  # nahodne poradi
  )
  for inputs, targets in train_loader:
      # inputs ... shape (100, 32, 32, 3)
      # targets ... shape (100,)
      ...
  ```

### TODO: implementujte metodu [`ans.data.BatchLoader.__iter__`](../ans/data.py).

In [7]:
train_loader = ans.data.BatchLoader(
    train_dataset,
    batch_size = 5,
    shuffle = True,
)
train_loader

BatchLoader:
    num_batches: 10000
    batch_size: 5

In [8]:
inputs, targets = next(iter(train_loader))
print(type(inputs), inputs.shape, inputs.dtype, inputs.min(), inputs.max())
print(type(targets), targets.shape, targets.dtype, targets.min(), targets.max())

<class 'torch.Tensor'> torch.Size([5, 3072]) torch.float32 tensor(0.) tensor(1.)
<class 'torch.Tensor'> torch.Size([5]) torch.int64 tensor(1) tensor(6)


In [9]:
test_linear_classification.TestBatchLoader.eval()

test_batch_size_even (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_batch_size_even) ... ok
test_batch_size_uneven (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_batch_size_uneven) ... ok
test_defaults (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_defaults) ... ok
test_output_tuple_supervised (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_output_tuple_supervised) ... ok
test_output_tuple_unsupervised (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_output_tuple_unsupervised) ... ok
test_shuffle (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_shuffle) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.012s

OK


<unittest.runner.TextTestResult run=6 errors=0 failures=0>

# Lineární softmax klasifikátor

- Lineární softmax klasifikátor optimalizuje křížovou entropii
  $$
  l_n = -\log\frac{\exp{s_{n,y_n}}}{\sum_{k=1}^{K}{\exp{s_{n,k}}}}
  $$
  kde skóre $\boldsymbol{s}_n$ jsou vypočteny jako lineární (afinní) funkce
  $$
  \boldsymbol{s}_n = \boldsymbol{x}_n \cdot \boldsymbol{w} + \boldsymbol{b}
  $$
- Klasifikátor bude implementován jako třída [`ans.classification.LinearSoftmaxModel`](../ans/classification.py).
- Třída obsahuje parametry klasifikátoru jako atributy  
  | atribut  | značení                               | rozměr       |
  |----------|---------------------------------------|--------------|
  | `weight` | $\boldsymbol{w} = [w_{d,k}]$          | $D \times K$ |
  | `bias`   | $\boldsymbol{b} = [b_1, \ldots, b_K]$ | $K$          |
- Metoda `__init__` inicializuje váhovou matici a vektor biasů na náhodná čísla  
  ``` python
  def __init__(self, in_size, out_size, weight_scale=1e-3) -> None:
    # self.weight = ...
    # self.bias = ...
  ```
- Metoda `train_step` provede jeden krok učení (včetně updatu parametrů) na zadané (mini)dávce a vrátí vypočtený loss a predikované skóre  
  ``` python
  def train_step(self, inputs, targets, learning_rate=1e-3):
    # predikce skore na davce
    # vypocet lossu (softmax krizova entropie)
    # vypocet gradientu
    # update parametru metodou nejvetsiho spadu
    # return loss, skore
  ```
- Metoda `val_step` provede pouze predikci (tedy bez updatu parametrů) na zadané (mini)dávce a vrátí vypočtený loss a predikované skóre  
  ``` python
  def val_step(self, inputs, targets, learning_rate=1e-3):
    # predikce skore na davce
    # vypocet lossu (softmax krizova entropie)
    # return loss, skore
  ```

**Poznámky k `train_step`**

- Data v tensorech jsou uložena *po řádcích*. Výpočet skóre je proto oproti přednášce *transponovaný*, tj.
  $$\boldsymbol{s}_n = \boldsymbol{x}_n \cdot \boldsymbol{w} + \boldsymbol{b}$$
  kde
  - $\boldsymbol{s}_n = [s_{n,1},\ldots,s_{n,K}]$ je *řádkový* vektor skóre pro $n$-tý vzorek (obrázek) $\boldsymbol{x}_n$ a každou z $K$ tříd
  - $\boldsymbol{x}_n = [x_{n,1},\ldots,x_{n,D}]$ je $n$-tý vzorek (obrázek) v dávce reprezentovaný jako (řádkový) vektor s rozměrem $D$.
- Celkový loss pro dávku vypočtěte jako *průměr* dílčích lossů za jednotlivé obrázky
  $$
  l = \frac{1}{N} \cdot \sum_{n=1}^{N}{l_n}
  $$
- Parametry updatujte základní metodou největšího spádu. Nepoužívejte momentum ani adaptivní metody.

### TODO: implementuje metodu [`ans.classification.LinearSoftmaxModel.__init__`](../ans/classification.py).

In [10]:
test_linear_classification.TestInit.eval()

test_init_params (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_init_params) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.LinearSoftmaxModel.train_step`](../ans/classification.py).

In [11]:
test_linear_classification.TestTrainStepSoftmax.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.010s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.LinearSoftmaxModel.val_step`](../ans/classification.py).

In [12]:
test_linear_classification.TestValStepSoftmax.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.006s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Úspěšnost: accuracy

- Kromě lossu budeme pro lepší orientaci měřit i přesnost (accuracy), byť tuto veličinu nebudeme přímo optimalizovat.
- Accuracy (správnost) $a$ vypočteme jako
  $$
  a = \frac{1}{N} \cdot \sum_{n=1}^{N}{\mathbb{1}(\hat{y}_n = y_n)}
  $$

### TODO: implementujte funkci [`ans.classification.accuracy`](../ans/classification.py).

In [13]:
test_linear_classification.TestAccuracy.eval()

test_accuracy (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_accuracy) ... ok
test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Trénování klasifikátoru

## Funkce `train_epoch`

- Vytvořímě funkci `train_epoch`, která převezme model a dataloader a zavolá `train_step` modelu na včechny dávky v dataloaderu.  
  Funkce musí umět převzít `learning_rate` a předat jí do volané `train_step`.

### TODO: implementujte funkci [`ans.classification.train_epoch`](../ans/classification.py).

In [14]:
 test_linear_classification.TestTrainEpoch.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.007s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Funkce `validate`

- Analogicky k ní vytvoříme ještě druhou funkci `validate`, která provede pouze predikci modelu na dataloaderu a vrátí průměrný loss a accuracy.

### TODO: implementujte funkci [`ans.classification.validate`](../ans/classification.py).

In [15]:
test_linear_classification.TestValidate.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.006s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Hlavní cyklus trénování

- Finálním úkolem je natrénovat lineární klasifikátor na co největší *validační* accuracy.
- **Nejlepší natrénovaný model uložte do souboru  `output/linear_softmax_weights.pt`**.
- Ukládejte pouze parametry `weights` a `bias`, tj. nikoliv celý objekt `LinearSoftmaxModel` a to následovně:
  ``` python
  torch.save(
    dict(weight=model.weight, bias=model.bias),
    "../output/linear_classification_softmax_weights.pt"
  )
  ```
- Model se později načte a v rámci testů vyhodnotí na validační sadě CIFAR-10. Je proto nutné dodržet cesty i formát ukládání.

### TODO: Natrénujte lineární softmax klasifikátor tak, aby dosáhl alespoň 40 % *validační* accuracy.

In [23]:
ans.utils.seed_everything(0)

# hyperparametry
batch_size = 1024
learning_rate = 0.0225
num_epochs = 35

# dataset
train_dataset = torchvision.datasets.CIFAR10(root="../data", train=True, download=True, transform=preprocess)
val_dataset = torchvision.datasets.CIFAR10(root="../data", train=False, download=True, transform=preprocess)

sample_img, _ = train_dataset[0]
in_size = sample_img.numel()  # 3 * 32 * 32 = 3072
out_size = len(train_dataset.classes)

# loadery
train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False)

# model
model = ans.classification.LinearSoftmaxModel(in_size=in_size, out_size=out_size, weight_scale=1e-3)

# validace pred trenovanim (sanity check, loss by mel byt cca 2.30)
train_loss, train_acc = ans.classification.validate(model, train_loader)
val_loss, val_acc = ans.classification.validate(model, val_loader)
print(f"after init: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}")
best_acc = 0.0
# optimalizace
for epoch in range(num_epochs):
    # trenovani
    ...

    # validace
    train_loss, train_acc = ans.classification.train_epoch(model, train_loader, learning_rate=learning_rate)
    val_loss, val_acc = ans.classification.train_epoch(model, val_loader, learning_rate=learning_rate)
    print(
        f"epoch {epoch + 1}: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_acc :
        best_acc = val_acc
# batch 1024, alpha 1/100, epoch 20 -> 0.3857800543308258
# batch 1024, alpha 1/50, epoch 30 -> 0.3997369259595871
# batch 1024, alpha = 0.0225, epoch 30 -> 0.4063616067171097 (Kolísá, alpha ideální bude nejspíše v rozmezí (0.02,0.0225)
print(best_acc)
os.makedirs("../output", exist_ok=True)
torch.save(dict(weight=model.weight, bias=model.bias), "../output/linear_classification_softmax_weights.pt")
print("Model byl úspěšně uložen!")

after init: train_loss=2.30397, train_acc=0.106, val_loss=2.30405, val_acc=0.105
epoch 1: train_loss=2.12832, train_acc=0.224, val_loss=2.00333, val_acc=0.296
epoch 2: train_loss=1.96359, train_acc=0.313, val_loss=1.93250, val_acc=0.325
epoch 3: train_loss=1.92115, train_acc=0.327, val_loss=1.88334, val_acc=0.347
epoch 4: train_loss=1.88181, train_acc=0.346, val_loss=1.85806, val_acc=0.358
epoch 5: train_loss=1.85732, train_acc=0.357, val_loss=1.84576, val_acc=0.359
epoch 6: train_loss=1.84954, train_acc=0.358, val_loss=1.83000, val_acc=0.366
epoch 7: train_loss=1.83478, train_acc=0.364, val_loss=1.82564, val_acc=0.368
epoch 8: train_loss=1.82409, train_acc=0.369, val_loss=1.81180, val_acc=0.371
epoch 9: train_loss=1.81589, train_acc=0.372, val_loss=1.81399, val_acc=0.365
epoch 10: train_loss=1.80648, train_acc=0.375, val_loss=1.79678, val_acc=0.376
epoch 11: train_loss=1.80535, train_acc=0.375, val_loss=1.79154, val_acc=0.378
epoch 12: train_loss=1.79812, train_acc=0.378, val_loss=1.7

In [24]:
test_linear_classification.TestSoftmaxValAccuracy.eval(preprocess_fn=preprocess)

test_softmax_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_softmax_val_acc) ... FAIL

FAIL: test_softmax_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_softmax_val_acc)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\tests\test_linear_classification.py", line 336, in test_softmax_val_acc
    self.assertGreaterEqual(val_acc, 0.4)
AssertionError: 0.3923 not greater than or equal to 0.4

----------------------------------------------------------------------
Ran 1 test in 6.149s

FAILED (failures=1)


<unittest.runner.TextTestResult run=1 errors=0 failures=1>

# Support Vector Machine (SVM)

- SVM je softmaxu velmi podobné. Z pohledu neuronových sítí se liší pouze způsobem výpočtu lossu.
- Místo (softmax) cross entropy optimalizuje hinge loss
  $$l_n = \sum_{c\ne y_n}\max(0, 1 + s_{n,c} - s_{n,y_n})$$
  kde:
  - $y_n \in \{1, \ldots, C\}$ je správný index třídy (celé číslo) na vzorku (obrázku) $\boldsymbol{x}_n$
  - $s_{n,i}$ je skóre (logity) predikované lineárním klasifikátorem pro $n$-tý obrázek a $i$-tou třídu.
- Gradient na $c$-tý sloupec $\boldsymbol{w}_{:,c}$ váhové matice $\boldsymbol{w}$ pak je
  $$
  \frac{\partial l_n}{\partial \boldsymbol{w}_{:,c}} = \begin{cases}
      -\sum_{c\ne y_n}\mathbb{1}(1 + s_{n,c} - s_{n,y_n} > 0)\cdot\boldsymbol{x}_n & \textrm{pokud} & c = y_n \\
      \mathbb{1}(1 + s_{n,c} - s_{n,y_n} > 0) \cdot \boldsymbol{x}_n & \textrm{pokud} & c \ne y_n
  \end{cases}
  $$
  kde
  - $\mathbb{1}(\cdot) = 1$, pokud podmínka $\cdot$ je splněna, jinak $\mathbb{1}(\cdot) = 0$
- Gradient na bias je
  $$
  \frac{\partial l_n}{\partial b_c} = \begin{cases}
      -\sum_{c\ne y_n}\mathbb{1}(1 + s_{n,c} - s_{n,y_n} > 0) & \textrm{pokud} & c = y_n \\
      \mathbb{1}(1 + s_{n,c} - s_{n,y_n} > 0) & \textrm{pokud} & c \ne y_n
  \end{cases}
  $$

### TODO: implementuje funkci [`ans.classification.LinearSVMModel.train_step`](../ans/classification.py).

In [25]:
test_linear_classification.TestTrainStepSVM.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... FAIL

FAIL: test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\tests\test_linear_classification.py", line 364, in test_step
    self.assertTensorsClose(loss, expected_loss)
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\tests\__init__.py", line 113, in assertTensorsClose
    self.fail(msg=f"{msg}\n{err_str}")
AssertionError: 
Scalars are not close!

Expected 1.161389946937561 but got 1.1654562950134277.
Absolute difference: 0.004066348075866699 (up to 0.0001 allowed)
Relative difference: 0.0035012771434686056 (up to 0.001 allowed)

Actual
1.1654562950134277

Expected
1.161389946937561


----------------------------------------------------------

<unittest.runner.TextTestResult run=2 errors=0 failures=1>

### TODO: implementuje funkci [`ans.classification.LinearSVMModel.val_step`](../ans/classification.py).

In [26]:
test_linear_classification.TestValStepSVM.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.005s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Hlavní cyklus trénování

- Nejlepší model uložte jako
  ``` python
  torch.save(
    dict(weight=model.weight, bias=model.bias),
    "../output/linear_classification_svm_weights.pt"
  )
  ```

### TODO: Natrénujte lineární SVM klasifikátor tak, aby dosáhl alespoň 39 % *validační* accuracy.

In [27]:
ans.utils.seed_everything(0)

# hyperparametry
batch_size = 256
learning_rate = 0.0015
num_epochs = 80

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# dataset
train_dataset = torchvision.datasets.CIFAR10(root="../data", train=True, transform=preprocess)
val_dataset = torchvision.datasets.CIFAR10(root="../data", train=False, transform=preprocess)

sample_img, _ = train_dataset[0]
in_size = sample_img.numel()  # 3 * 32 * 32 = 3072
out_size = len(train_dataset.classes)

# loadery (with device passed directly)
train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True, device=device)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False, device=device)

# model (použití in_size a out_size)
model = ans.classification.LinearSVMModel(in_size=in_size, out_size=out_size, weight_scale=1e-3)
model.to(device)

# validace pred trenovanim (sanity check, loss u SVM by měl být cca 9)
train_loss, train_acc = ans.classification.validate(model, train_loader)
val_loss, val_acc = ans.classification.validate(model, val_loader)
print(f"after init: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}")

best_acc = 0.0

# optimalizace
for epoch in range(num_epochs):
    # trenovani
    ans.classification.train_epoch(model, train_loader, learning_rate=learning_rate)

    # validace
    train_loss, train_acc = ans.classification.validate(model, train_loader)
    val_loss, val_acc = ans.classification.validate(model, val_loader)
    
    print(
        f"epoch {epoch + 1}: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
# batch = 1024, alpha = 0.021, epoch = 30 -> 0.2971
# batch = 1024, alpha = 0.001, epoch = 30 -> 0.3703
# batch = 1024, alpha = 0.001, epoch = 40 -> 0.3739
# batch = 1024, alpha = 0.002, epoch = 40, reg *5 -> 0.3826
# batch = 256, alpha = 0.001, epoch = 60, reg *5 -> 0.3889
# batch = 256, alpha = 0.0015, epoch = 80, reg *5 -> 0.392

print(best_acc)
os.makedirs("../output", exist_ok=True)
torch.save(dict(weight=model.weight, bias=model.bias), "../output/linear_classification_svm_weights.pt")
print("Model byl úspěšně uložen!")

after init: train_loss=9.00865, train_acc=0.106, val_loss=9.00949, val_acc=0.105
epoch 1: train_loss=5.38092, train_acc=0.325, val_loss=5.41717, val_acc=0.322
epoch 2: train_loss=5.28357, train_acc=0.320, val_loss=5.31489, val_acc=0.314
epoch 3: train_loss=5.05916, train_acc=0.323, val_loss=5.11791, val_acc=0.321
epoch 4: train_loss=4.96977, train_acc=0.336, val_loss=5.02045, val_acc=0.326
epoch 5: train_loss=4.71587, train_acc=0.357, val_loss=4.77087, val_acc=0.350
epoch 6: train_loss=4.86365, train_acc=0.340, val_loss=4.93628, val_acc=0.332
epoch 7: train_loss=4.72146, train_acc=0.365, val_loss=4.78319, val_acc=0.361
epoch 8: train_loss=4.60014, train_acc=0.373, val_loss=4.68514, val_acc=0.367
epoch 9: train_loss=4.74491, train_acc=0.354, val_loss=4.82696, val_acc=0.344
epoch 10: train_loss=4.79477, train_acc=0.338, val_loss=4.86690, val_acc=0.333
epoch 11: train_loss=4.56574, train_acc=0.379, val_loss=4.65774, val_acc=0.370
epoch 12: train_loss=4.51160, train_acc=0.376, val_loss=4.6

In [28]:
test_linear_classification.TestSVMValAccuracy.eval(preprocess_fn=preprocess)

test_svm_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_svm_val_acc) ... ok

----------------------------------------------------------------------
Ran 1 test in 5.845s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>